In [ ]:
# app/services/error_utils.py

import json
import logging
import re
from typing import Any

from app.models.schemas import ImageAnalysisResult, ReviewAnalysisResult, ReviewEvidence

logger = logging.getLogger(__name__)


class ExternalAPIError(ValueError):
    """Raised when an external API returns an error response."""


def validate_google_places_status(payload: dict[str, Any], context: str) -> None:
    """
    Validate Google Places API response status.

    This handles:
    - invalid API key
    - quota exceeded
    - request denied
    - invalid request
    - unknown API errors

    ZERO_RESULTS is not treated as an exception because it is a valid empty result.
    """
    status = payload.get("status", "UNKNOWN_ERROR")
    error_message = payload.get("error_message", "")

    if status in {"OK", "ZERO_RESULTS"}:
        return

    friendly_messages = {
        "REQUEST_DENIED": "Google Places request was denied. Please check the Google Maps API key and make sure the Places API is enabled.",
        "OVER_QUERY_LIMIT": "Google Places quota was exceeded. Please check API quota or try again later.",
        "INVALID_REQUEST": "Google Places request was invalid. Please check the request parameters.",
        "UNKNOWN_ERROR": "Google Places returned a temporary server error. Please try again later.",
    }

    message = friendly_messages.get(status, f"Google Places request failed with status: {status}")

    if error_message:
        message = f"{message} Details: {error_message}"

    raise ExternalAPIError(f"{context}: {message}")


def empty_restaurant_response_note(zip_code: str, cuisine: str) -> str:
    """
    Return a clear message when no restaurants are found.
    """
    return (
        f"No restaurants were found for cuisine='{cuisine}' near ZIP code='{zip_code}'. "
        "Try a broader cuisine term, a different ZIP code, or a less restrictive search."
    )


def fallback_review_analysis(
    evidence: list[ReviewEvidence] | None = None,
    reason: str = "Review analysis was unavailable.",
) -> ReviewAnalysisResult:
    """
    Return a safe fallback review analysis.

    This handles:
    - no reviews
    - Gemini review analysis failure
    - malformed Gemini JSON
    """
    evidence = evidence or []

    if not evidence:
        return ReviewAnalysisResult(
            signature_dishes=[],
            service="Unknown",
            value="Unknown",
            wait_impression="Unknown",
            vibe="Unknown",
            pros=["No review evidence was available."],
            cons=[],
            evidence=[],
        )

    return ReviewAnalysisResult(
        signature_dishes=[],
        service="Unknown",
        value="Unknown",
        wait_impression="Unknown",
        vibe="Unknown",
        pros=[reason],
        cons=[],
        evidence=evidence,
    )


def fallback_image_analysis(reason: str = "Image analysis was unavailable.") -> ImageAnalysisResult:
    """
    Return a safe fallback image analysis.

    This handles:
    - no photos
    - image URL fetch failure
    - Gemini VLM failure
    """
    return ImageAnalysisResult(
        visual_vibe="unknown",
        space_impression="unknown",
        food_visual_cues=[],
        group_suitability="unknown",
        visual_confidence="low",
        image_evidence_summary=reason,
    )


def safe_parse_json_object(raw_text: str) -> dict[str, Any]:
    """
    Safely parse a Gemini JSON response.

    Handles:
    - normal JSON
    - JSON wrapped in ```json code fences
    - invalid JSON
    - JSON that is not an object
    """
    trimmed = raw_text.strip()

    if trimmed.startswith("```"):
        trimmed = re.sub(r"^```(?:json)?", "", trimmed).strip()
        trimmed = re.sub(r"```$", "", trimmed).strip()

    try:
        parsed = json.loads(trimmed)
    except json.JSONDecodeError as exc:
        logger.exception("Malformed JSON response from Gemini: %s", trimmed[:1000])
        raise ValueError("Gemini response was not valid JSON.") from exc

    if not isinstance(parsed, dict):
        raise ValueError("Gemini response must be a JSON object.")

    return parsed